In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
len(text)

1115394

In [4]:
text[:1000]  # Display the first 1000 characters of the text

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [5]:
chars=sorted(list(set(text)))
#chars
vocab_size = len(chars)
print(''.join(chars))
print(f'Vocab size: {vocab_size}')


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


In [6]:
#mapping from characters to integers
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}

encode= lambda s: [stoi[c] for c in s]  
# encoder: take a string, output a list of integers
decode= lambda l: ''.join([itos[i] for i in l])  
# decoder: take a list of integers, output a string

print(encode("fuck"))
print(decode(encode("fuck")))

[44, 59, 41, 49]
fuck


In [7]:
import torch
data=torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])  # first 1000 characters as integers


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
#train / validation sets
n=int(0.9*len(data))  # first 90% will be train, rest val
train_data=data[:n]
val_data=data[n:]

In [9]:
block_size=8  # how many characters to consider for predictions
train_data[:block_size+1]  # first block_size+1 characters

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x=train_data[:block_size]  # first block_size characters
y=train_data[1:block_size+1]  # the targets, shifted by one character
print(x)
print(y)
print(x[0])
print(x[0:1])
print(x[1:2])
for t in range(block_size):
    #t=0~7
    context=x[:t+1] 
    #左闭右开，x[:t+1],索引：0~t共t+1个int组成的list
    target=y[t]  # predict the next character
    #target=x[t+1]
    print(f"when input is {context} the target: {target}")


tensor([18, 47, 56, 57, 58,  1, 15, 47])
tensor([47, 56, 57, 58,  1, 15, 47, 58])
tensor(18)
tensor([18])
tensor([47])
when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [11]:
torch.manual_seed(1337)
batch_size=4  # how many independent sequences will we process in parallel?
block_size=8  # what is the maximum context length for predictions? 

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data=train_data if split=='train' else val_data
    ix=torch.randint(len(data)-block_size, (batch_size,))
    #ix:[4,56,54,2],block的随机起始位置
    x=torch.stack([data[i:i+block_size] for i in ix])
    y=torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb,yb=get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

for b in range(batch_size):#batch dimension
    for t in range(block_size):# time dimension
        context=xb[b,:t+1]
        target=yb[b,t]
        print(f"when input is {context.tolist()} the target: {target.item()}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53, 56,

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :]  # becomes (B,C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B,C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B,1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B,T+1)
        return idx

m=BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

NameError: name 'vocab_size' is not defined

In [13]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

KeyboardInterrupt: 

In [20]:

print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


IOr with is aras Tht; thap
YORENGind hathinthethasiouro oulof I and s l SS:
Iffof; hat t-asoresere ashath fover;

AUMENGHave ie? nds to wisus, athal
Fiotha her owa
Fouidif toury aris ior yoress ane hit in,
O:
LETAUns.
Isat t tst far thas s sthasers t Bokerdace
My e
TENIris,
G oue, hon buime.
adive m


In [15]:
torch.manual_seed(1337)
B,T,C=4,8,2
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [24]:
# x[b,t]=mean_(i<=t)x[b,i]
xbow= torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev=x[b,:t+1]  # (t,C)
        xbow[b,t]=torch.mean(xprev, 0)  # (C)#按行平均

In [25]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [26]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [27]:
wei=torch.tril(torch.ones(T,T))#下三角矩阵
wei=wei/wei.sum(1, keepdim=True)#保留维度->广播机制
xbow2=wei@x #(T,T)@(B,T,C) ->(B,T,T)@(B,T,C)-> (B,T,C)
torch.allclose(xbow, xbow2)

False

In [29]:
xbow[0],xbow2[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [30]:
diff = (xbow - xbow2).abs()

print("max diff:", diff.max())
print("mean diff:", diff.mean())

torch.set_printoptions(precision=10)
print(xbow[0])
print(xbow2[0])

max diff: tensor(3.2363e-08)
mean diff: tensor(7.2141e-09)
tensor([[ 0.1807715893, -0.0699880943],
        [-0.0894259512, -0.4925962687],
        [ 0.1489711404, -0.3198941946],
        [ 0.3503567576, -0.2238335162],
        [ 0.3525155187,  0.0545087978],
        [ 0.0687807426, -0.0396054536],
        [ 0.0926631615, -0.0682015866],
        [-0.0340590850,  0.1332357526]])
tensor([[ 0.1807715893, -0.0699880943],
        [-0.0894259512, -0.4925962687],
        [ 0.1489711404, -0.3198942244],
        [ 0.3503567576, -0.2238335162],
        [ 0.3525155187,  0.0545087904],
        [ 0.0687807500, -0.0396054536],
        [ 0.0926631540, -0.0682015717],
        [-0.0340590850,  0.1332357526]])


In [32]:
#version 3：use softmax
tril=torch.tril(torch.ones(T,T))#下三角矩阵
wei=torch.zeros((T,T))
wei=wei.masked_fill(tril==0, float('-inf'))  # set upper triangular part to -inf
wei=F.softmax(wei, dim=-1)  # apply softmax to get weights
xbow3=wei@x #(T,T)@(B,T,C) ->(B,T,T)@(B,T,C)-> (B,T,C)
torch.allclose(xbow2, xbow3)

True

In [ ]:
#version 4: use self-attention
import torch
import torch.nn as nn

torch.manual_seed(1337)
B,T,C=4,8,32
x=torch.randn(B,T,C)

#a single Head performing self-attention
head_size=16
key=nn.Linear(C, head_size, bias=False)
query=nn.Linear(C, head_size, bias=False)
value=nn.Linear(C, head_size, bias=False)
k=key(x) #(B,T,16)
q=query(x) #(B,T,16)#
#wei=q@k.transpose(-2,-1) #(B,T,16)@(B,16,T)->(B,T,T
wei=q@k.transpose(-2,-1)*(head_size**-0.5) #(B,T,16)@(B,16,T)->(B,T,T)
#根号dk

tril=torch.tril(torch.ones(T,T))#下三角矩阵
#wei=torch.zeros((T,T))
wei=wei.masked_fill(tril==0, float('-inf'))  # set upper

v=value(x) #(B,T,16)
out=wei@v #(B,T,T)@(B,T,16)->(B,T,16)
#out=wei@x

out.shape


torch.Size([4, 8, 16])